In [6]:
ython -m spacy download en_core_web_sm

SyntaxError: invalid syntax (2968366042.py, line 1)

In [7]:
import spacy
import re
from sklearn.metrics import precision_recall_fscore_support

nlp = spacy.load("en_core_web_sm")

texts = [
    "Sundar Pichai visited Microsoft in New York.",
    "Narendra Modi met officials in New Delhi.",
    "Google opened a new office in London.",
    "Elon Musk works at Tesla in California."
]

actual = [
    ["PERSON", "ORG", "GPE"],
    ["PERSON", "GPE"],
    ["ORG", "GPE"],
    ["PERSON", "ORG", "GPE"]
]

def preprocess(text):
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def rule_based(text):
    entities = []
    pattern = r'\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b'

    for word in re.findall(pattern, text):
        if word in ["New York", "New Delhi", "London", "California"]:
            entities.append((word, "GPE"))
        elif word in ["Google", "Microsoft", "Tesla"]:
            entities.append((word, "ORG"))
        else:
            entities.append((word, "PERSON"))

    return entities

def statistical_ner(text):
    doc = nlp(text)
    return [(ent.text, ent.label_) for ent in doc.ents]

def evaluate(predicted, actual):
    correct = 0

    for p in predicted:
        if p in actual:
            correct += 1
            actual.remove(p)

    precision = correct / (correct + len(predicted) - correct) if predicted else 0
    recall = correct / (correct + len(actual)) if correct + len(actual) else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0

    return precision, recall, f1

for text, true_entities in zip(texts, actual):
    text = preprocess(text)

    print("\nText:", text)

    rule_entities = rule_based(text)
    spacy_entities = statistical_ner(text)

    print("Rule-based:", rule_entities)
    print("SpaCy:", spacy_entities)

    print("Actual labels:", true_entities)

    true_labels = [
        (ent.text, ent.label_)
        for ent in nlp(text).ents
    ]

    print("SpaCy entity count:", len(true_labels))

print("\nNER using SpaCy:")

for text in texts:
    doc = nlp(preprocess(text))
    for ent in doc.ents:
        print(ent.text, ":", ent.label_)


Text: Sundar Pichai visited Microsoft in New York.
Rule-based: [('Sundar Pichai', 'PERSON'), ('Microsoft', 'ORG'), ('New York', 'GPE')]
SpaCy: [('Sundar Pichai', 'ORG'), ('Microsoft', 'ORG'), ('New York', 'GPE')]
Actual labels: ['PERSON', 'ORG', 'GPE']
SpaCy entity count: 3

Text: Narendra Modi met officials in New Delhi.
Rule-based: [('Narendra Modi', 'PERSON'), ('New Delhi', 'GPE')]
SpaCy: [('Narendra Modi', 'PERSON'), ('New Delhi', 'GPE')]
Actual labels: ['PERSON', 'GPE']
SpaCy entity count: 2

Text: Google opened a new office in London.
Rule-based: [('Google', 'ORG'), ('London', 'GPE')]
SpaCy: [('Google', 'ORG'), ('London', 'GPE')]
Actual labels: ['ORG', 'GPE']
SpaCy entity count: 2

Text: Elon Musk works at Tesla in California.
Rule-based: [('Elon Musk', 'PERSON'), ('Tesla', 'ORG'), ('California', 'GPE')]
SpaCy: [('Elon Musk', 'PERSON'), ('Tesla', 'ORG'), ('California', 'GPE')]
Actual labels: ['PERSON', 'ORG', 'GPE']
SpaCy entity count: 3

NER using SpaCy:
Sundar Pichai : ORG
Mic

In [8]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_true = ["PERSON", "ORG", "GPE", "PERSON", "GPE"]
y_pred = ["PERSON", "ORG", "GPE", "ORG", "GPE"]

print("Precision:", precision_score(
    y_true, y_pred, average="micro", zero_division=0
))
print("Recall:", recall_score(
    y_true, y_pred, average="micro", zero_division=0
))
print("F1-score:", f1_score(
    y_true, y_pred, average="micro", zero_division=0
))

Precision: 0.8
Recall: 0.8
F1-score: 0.8
